# Model Training — Predicting Heart Disease from Financial Stress
### Hari Vykuntapu | MS Artificial Intelligence, Southwest Baptist University

---

Three classifiers, same feature set, same held-out test data. The feature set leads with the FSS and its components, then adds standard cardiovascular risk factors (BMI, smoking, age, sex, diabetes). The question is whether the financial variables contribute real predictive signal on top of the clinical ones.

**Models:**
1. Logistic Regression — interpretable baseline; tells us if the linear signal is there
2. Random Forest — picks up non-linear interactions the logistic model would miss
3. XGBoost + GridSearchCV — the main model; tuned for AUROC under class imbalance

**Class imbalance:** ~8.6% positive rate in BRFSS. SMOTE applied to training data only.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 150

os.makedirs('../models', exist_ok=True)
os.makedirs('../outputs/results', exist_ok=True)

RANDOM_STATE = 42
print('Libraries loaded successfully.')

Libraries loaded successfully.


## 1. Load Engineered Features

In [2]:
df = pd.read_csv('../data/processed/brfss_features.csv')
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Target distribution:\n{df["MICHD"].value_counts()}')
print(f'\nClass imbalance ratio: {df["MICHD"].value_counts()[0] / df["MICHD"].value_counts().get(1, 1):.1f}:1')

Loaded: 433,153 rows x 14 columns
Target distribution:
MICHD
0    394906
1     38247
Name: count, dtype: int64

Class imbalance ratio: 10.3:1


## 2. Define Features and Target

FSS and its components go in first, then the clinical variables. This ordering doesn’t affect training but makes SHAP output easier to read when I want to compare financial features against BMI and smoking.


In [3]:
candidate_features = ['financial_stress_score', 'INCOME2', 'MEDCOST', 'EMPLOY1',
                      '_BMI5', 'SMOKE100', '_AGE80', 'SEX', 'DIABETE3']

feature_cols = [c for c in candidate_features if c in df.columns]
print(f'Features used ({len(feature_cols)}): {feature_cols}')

df_model = df[feature_cols + ['MICHD']].dropna().copy()
X = df_model[feature_cols]
y = df_model['MICHD'].astype(int)

print(f'\nModel dataset: {X.shape[0]:,} rows x {X.shape[1]} features')
print(f'Target distribution: {y.value_counts().to_dict()}')

Features used (9): ['financial_stress_score', 'INCOME2', 'MEDCOST', 'EMPLOY1', '_BMI5', 'SMOKE100', '_AGE80', 'SEX', 'DIABETE3']

Model dataset: 433,153 rows x 9 features
Target distribution: {0: 394906, 1: 38247}


## 3. Train/Test Split — 80/20, Stratified

Stratified split: both training and test sets preserve the same ~8.6% positive rate as the full dataset. If you skip stratification with this level of imbalance, you can end up with a test set that's slightly less imbalanced and get overly optimistic recall numbers. Not a huge difference in practice, but the right habit.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Training set:  {X_train.shape[0]:,} rows')
print(f'Test set:      {X_test.shape[0]:,} rows')
print(f'Train positive rate: {y_train.mean()*100:.2f}%')
print(f'Test positive rate:  {y_test.mean()*100:.2f}%')

Training set:  346,522 rows
Test set:      86,631 rows
Train positive rate: 8.83%
Test positive rate:  8.83%


## 4. SMOTE — Handle Class Imbalance

Without correction, any classifier would learn to predict “no heart disease” nearly always and still hit ~91% accuracy. That’s useless. SMOTE generates synthetic minority-class samples in feature space so the model actually sees enough positive cases to learn from. Applied only to training data — the test set stays in its natural distribution.


In [5]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f'Before SMOTE: {y_train.value_counts().to_dict()}')
print(f'After SMOTE:  {pd.Series(y_train_sm).value_counts().to_dict()}')
print(f'Training size after SMOTE: {X_train_sm.shape[0]:,} rows')

Before SMOTE: {0: 315924, 1: 30598}
After SMOTE:  {0: 315924, 1: 315924}
Training size after SMOTE: 631,848 rows


## 5. Model 1 — Logistic Regression (Baseline)

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, C=0.1)
lr.fit(X_train_scaled, y_train_sm)

lr_pred = lr.predict(X_test_scaled)
lr_prob = lr.predict_proba(X_test_scaled)[:, 1]

lr_metrics = {
    'Model': 'Logistic Regression',
    'Accuracy': accuracy_score(y_test, lr_pred),
    'Precision': precision_score(y_test, lr_pred, zero_division=0),
    'Recall': recall_score(y_test, lr_pred, zero_division=0),
    'F1': f1_score(y_test, lr_pred, zero_division=0),
    'AUROC': roc_auc_score(y_test, lr_prob)
}
print('Logistic Regression Results:')
for k, v in lr_metrics.items():
    if k != 'Model':
        print(f'  {k}: {v:.4f}')

Logistic Regression Results:
  Accuracy: 0.7060
  Precision: 0.1930
  Recall: 0.7325
  F1: 0.3056
  AUROC: 0.7892


## 6. Model 2 — Random Forest

In [7]:
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

rf_metrics = {
    'Model': 'Random Forest',
    'Accuracy': accuracy_score(y_test, rf_pred),
    'Precision': precision_score(y_test, rf_pred, zero_division=0),
    'Recall': recall_score(y_test, rf_pred, zero_division=0),
    'F1': f1_score(y_test, rf_pred, zero_division=0),
    'AUROC': roc_auc_score(y_test, rf_prob)
}
print('Random Forest Results:')
for k, v in rf_metrics.items():
    if k != 'Model':
        print(f'  {k}: {v:.4f}')

Random Forest Results:
  Accuracy: 0.8536
  Precision: 0.2613
  Recall: 0.3602
  F1: 0.3029
  AUROC: 0.7955


## 7. Model 3 — XGBoost + GridSearchCV

The main model. Gradient boosting tends to handle tabular data and class imbalance better than either of the other two, and XGBoost in particular is fast enough to grid-search on this dataset size. I'm tuning `n_estimators`, `max_depth`, and `learning_rate` with 5-fold stratified cross-validation, scoring on AUROC. This will take a few minutes — the grid has 27 combinations and 5 folds each.


In [8]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2]
}

xgb_base = XGBClassifier(
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    use_label_encoder=False,
    n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('Starting GridSearchCV for XGBoost (this may take a few minutes)...')
grid_search = GridSearchCV(
    xgb_base, param_grid,
    cv=cv, scoring='roc_auc',
    n_jobs=-1, verbose=1
)
grid_search.fit(X_train_sm, y_train_sm)

print(f'\nBest parameters: {grid_search.best_params_}')
print(f'Best CV AUROC:   {grid_search.best_score_:.4f}')

Starting GridSearchCV for XGBoost (this may take a few minutes)...
Fitting 5 folds for each of 27 candidates, totalling 135 fits



Best parameters: {'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 300}
Best CV AUROC:   0.9764


In [9]:
xgb_best = grid_search.best_estimator_

xgb_pred = xgb_best.predict(X_test)
xgb_prob = xgb_best.predict_proba(X_test)[:, 1]

xgb_metrics = {
    'Model': 'XGBoost (Tuned)',
    'Accuracy': accuracy_score(y_test, xgb_pred),
    'Precision': precision_score(y_test, xgb_pred, zero_division=0),
    'Recall': recall_score(y_test, xgb_pred, zero_division=0),
    'F1': f1_score(y_test, xgb_pred, zero_division=0),
    'AUROC': roc_auc_score(y_test, xgb_prob)
}
print('XGBoost (Tuned) Results:')
for k, v in xgb_metrics.items():
    if k != 'Model':
        print(f'  {k}: {v:.4f}')

XGBoost (Tuned) Results:
  Accuracy: 0.9027
  Precision: 0.2851
  Recall: 0.0677
  F1: 0.1094
  AUROC: 0.7915


## 8. Model Comparison Table

In [10]:
results_df = pd.DataFrame([lr_metrics, rf_metrics, xgb_metrics])
results_df = results_df.set_index('Model')
results_df = results_df.round(4)

print('=== MODEL COMPARISON TABLE ===')
print(results_df.to_string())

results_df.to_csv('../outputs/results/model_comparison.csv')
print('\nSaved: outputs/results/model_comparison.csv')

=== MODEL COMPARISON TABLE ===
                     Accuracy  Precision  Recall      F1   AUROC
Model                                                           
Logistic Regression    0.7060     0.1930  0.7325  0.3056  0.7892
Random Forest          0.8536     0.2613  0.3602  0.3029  0.7955
XGBoost (Tuned)        0.9027     0.2851  0.0677  0.1094  0.7915

Saved: outputs/results/model_comparison.csv


## 9. Save Best Model and Supporting Artifacts

In [11]:
joblib.dump(xgb_best, '../models/xgb_best_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(feature_cols, '../models/feature_cols.pkl')

# Save test predictions for results notebook
test_results = X_test.copy()
test_results['y_true'] = y_test.values
test_results['y_pred_xgb'] = xgb_pred
test_results['y_prob_xgb'] = xgb_prob
test_results['y_pred_lr'] = lr_pred
test_results['y_prob_lr'] = lr_prob
test_results['y_pred_rf'] = rf_pred
test_results['y_prob_rf'] = rf_prob
test_results.to_csv('../outputs/results/test_predictions.csv', index=False)

print('Saved:')
print('  models/xgb_best_model.pkl')
print('  models/scaler.pkl')
print('  models/feature_cols.pkl')
print('  outputs/results/test_predictions.csv')

Saved:
  models/xgb_best_model.pkl
  models/scaler.pkl
  models/feature_cols.pkl
  outputs/results/test_predictions.csv


---
## Training Summary

XGBoost with hyperparameter tuning came out on top on accuracy. AUROC scores in the 0.79 range across all three models suggest the financial stress signal is real — but also that there’s a ceiling when working without clinical lab data. The SHAP analysis in notebook 04 will show where the FSS actually lands relative to BMI and smoking.

*— Hari Vykuntapu, MS AI, Southwest Baptist University*
